In [ ]:
import ee

ee.Initialize(project="earth-engine-project-495404")

print("Earth Engine works in notebook!")

In [ ]:
%pip install --quiet earthengine-api pandas plotly

In [ ]:
import ee
import pandas as pd
import plotly.express as px

print("ee version:", ee.__version__)
print("pandas version:", pd.__version__)
print("plotly version:", px.__name__, "imported successfully")

In [ ]:
import ee

ee.Initialize(project="earth-engine-project-495404")

# Sanity check: ask Earth Engine to compute 7+8 on its servers
result = ee.Number(7).add(8).getInfo()
print("Earth Engine connected from notebook")
print("Sanity check (7+8):", result)

In [ ]:
districts = (
    ee.FeatureCollection("FAO/GAUL/2015/level2")
    .filter(ee.Filter.eq("ADM0_NAME", "Bangladesh"))
)

district_count = districts.size().getInfo()
print(f"Bangladesh districts loaded: {district_count}")

In [ ]:
rangpur = districts.filter(ee.Filter.eq("ADM2_NAME", "Rangpur")).first()
rangpur_geom = rangpur.geometry()

area_km2 = rangpur_geom.area().divide(1e6).getInfo()
centroid = rangpur_geom.centroid().coordinates().getInfo()

print(f"Rangpur area: {area_km2:.1f} km^2")
print(f"Rangpur centroid (lon, lat): {centroid[0]:.4f}, {centroid[1]:.4f}")

In [ ]:
end_date = ee.Date(pd.Timestamp.today().strftime("%Y-%m-%d"))
start_date = end_date.advance(-24, "month")

s2_raw = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(rangpur_geom)
    .filterDate(start_date, end_date)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
)

scene_count = s2_raw.size().getInfo()
print(f"Sentinel-2 scenes (raw, <60% cloud): {scene_count}")

In [ ]:
def mask_s2_clouds(image):
    qa = image.select("QA60")
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11
    mask = (
        qa.bitwiseAnd(cloud_bit).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit).eq(0))
    )
    return image.updateMask(mask).divide(10000).copyProperties(
        image, ["system:time_start"]
    )

s2 = s2_raw.map(mask_s2_clouds)
print("Cloud-masked collection ready")

In [ ]:
def add_ndvi(image):
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")
    return image.addBands(ndvi)

s2_ndvi = s2.map(add_ndvi)
print("NDVI band added")

In [ ]:
def monthly_ndvi(month_offset):
    month_start = end_date.advance(month_offset, "month")
    month_end = month_start.advance(1, "month")

    monthly_collection = (
        s2_ndvi
        .filterDate(month_start, month_end)
        .select("NDVI")
    )

    ndvi_value = ee.Algorithms.If(
        monthly_collection.size().gt(0),
        monthly_collection.mean().reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=rangpur_geom,
            scale=100,
            maxPixels=1e9,
        ).get("NDVI"),
        None,
    )

    return ee.Feature(None, {
        "date": month_start.format("YYYY-MM"),
        "ndvi": ndvi_value,
    })

monthly_features = ee.FeatureCollection(
    [monthly_ndvi(i) for i in range(-23, 1)]
)

print("Monthly composites defined (24 months, empty-month safe)")

In [ ]:
features = monthly_features.getInfo()["features"]

rows = [
    (f["properties"]["date"], f["properties"].get("ndvi"))
    for f in features
]

df = pd.DataFrame(rows, columns=["date", "ndvi"])
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print(df)
print(f"\nNon-null months: {df['ndvi'].notna().sum()} / {len(df)}")

In [ ]:
%pip install --quiet nbformat

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["date"],
    y=df["ndvi"],
    mode="lines+markers",
    name="NDVI",
    line=dict(color="#2E7D32", width=2),
    marker=dict(size=6),
    connectgaps=False,
))

fig.update_layout(
    title="Rangpur District — Monthly Mean NDVI (Sentinel-2)",
    xaxis_title="Month",
    yaxis_title="NDVI",
    yaxis=dict(range=[0, 1]),
    template="simple_white",
    height=420,
    margin=dict(l=60, r=30, t=60, b=50),
)

fig.show()

## Khulna validation

In [ ]:
def ndvi_timeseries(district_name, end_date, months=24):
    feature = districts.filter(ee.Filter.eq("ADM2_NAME", district_name)).first()
    geom = feature.geometry()

    start = end_date.advance(-(months - 1), "month")

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(geom)
        .filterDate(start, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
        .map(mask_s2_clouds)
        .map(add_ndvi)
    )

    rows = []
    for offset in range(-(months - 1), 1):
        m_start = end_date.advance(offset, "month")
        m_end = m_start.advance(1, "month")
        monthly_col = collection.filterDate(m_start, m_end).select("NDVI")

        date_str = m_start.format("YYYY-MM").getInfo()
        size_check = monthly_col.size().getInfo()

        if size_check == 0:
            rows.append((date_str, None))
            continue

        stats = monthly_col.mean().reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=100,
            maxPixels=1e9,
        ).getInfo()
        rows.append((date_str, stats.get("NDVI")))

    df = pd.DataFrame(rows, columns=["date", "ndvi"])
    df["date"] = pd.to_datetime(df["date"])
    return df.sort_values("date").reset_index(drop=True)

In [ ]:
khulna_df = ndvi_timeseries("Khulna", end_date)
print(khulna_df)
print(f"\nNon-null months: {khulna_df['ndvi'].notna().sum()} / {len(khulna_df)}")

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["date"], y=df["ndvi"],
    mode="lines+markers", name="Rangpur",
    line=dict(color="#2E7D32", width=2),
))

fig.add_trace(go.Scatter(
    x=khulna_df["date"], y=khulna_df["ndvi"],
    mode="lines+markers", name="Khulna",
    line=dict(color="#1565C0", width=2),
))

fig.update_layout(
    title="NDVI Comparison — Rangpur vs Khulna",
    xaxis_title="Month",
    yaxis_title="NDVI",
    yaxis=dict(range=[0, 1]),
    template="simple_white",
    height=420,
    margin=dict(l=60, r=30, t=60, b=50),
)

fig.show()